We used Google Collab for this project, since it offers the possibility to use a GPU for training purposes.
We try two different neural network architecture approaches with the purpose of obtaining the best result, using the same data in both cases.

## --- Section 1: Necessary imports and data downloading ---

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
# Import necessary libraries
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt
import numpy as np
import os
import datetime


In [ ]:
#Downloading the dataset from Kaggle
import kagglehub

# Download latest version
path = kagglehub.dataset_download("lukechugh/best-alzheimer-mri-dataset-99-accuracy")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/best-alzheimer-mri-dataset-99-accuracy


In [4]:
import os

#Checking the content of the main folder
print("Dataset path:", path)
print("\nContent of the main folder:")
print(os.listdir(path))


Dataset path: /kaggle/input/best-alzheimer-mri-dataset-99-accuracy

Content of the main folder:
['Combined Dataset']


In [ ]:
#Seeing what's inside of the "combined dataset"
print(os.listdir(os.path.join(path, "Combined Dataset"))) 

['test', 'train']


## --- Section 2: Data Augmentation & Preprocessing ---

In [ ]:

import os
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Paths to train and test folders from the Kaggle dataset
dataset_path = os.path.join(path, "Combined Dataset")
train_dir = os.path.join(dataset_path, "train")
test_dir = os.path.join(dataset_path, "test")

# Define image dimensions and batch size
img_height, img_width = 224, 224
batch_size = 32
epochs = 25

# Data augmentation for training + rescaling
datagen = ImageDataGenerator(
    rescale=1./255,        # Normalize pixel values to [0, 1]
    rotation_range=20,     # Randomly rotate images by up to 20 degrees
    width_shift_range=0.2, # Randomly shift images horizontally
    height_shift_range=0.2,# Randomly shift images vertically
    horizontal_flip=True,  # Randomly flip images horizontally
    validation_split=0.2   # Split training set into train/validation
)

# Train data
train_generator = datagen.flow_from_directory(
    train_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='categorical',
    subset='training'
)

# Validation data
validation_generator = datagen.flow_from_directory(
    train_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation'
)

# Test data (no augmentation, only rescaling)
test_datagen = ImageDataGenerator(rescale=1./255)
test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)

# Get class names and number of classes
class_names = list(train_generator.class_indices.keys())
print("Class labels:", class_names)
num_classes = len(class_names)


Found 8192 images belonging to 4 classes.
Found 2048 images belonging to 4 classes.
Found 1279 images belonging to 4 classes.
Class labels: ['Mild Impairment', 'Moderate Impairment', 'No Impairment', 'Very Mild Impairment']


## First Architecture

In [10]:

# --- Section 3: Build the Multi-Layer CNN Model ---

model = Sequential([
    # First Convolutional Block
    Conv2D(32, (3, 3), activation='relu', input_shape=(img_height, img_width, 3)),
    MaxPooling2D(pool_size=(2, 2)),

    # Second Convolutional Block
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),

    # Third Convolutional Block
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),

    # Flatten the 3D output to 1D for the dense layers
    Flatten(),

    # Fully Connected Layers
    Dense(512, activation='relu'),
    Dropout(0.5), # Dropout layer to reduce overfitting

    # Final output layer with SoftMax activation for multi-class classification
    Dense(num_classes, activation='softmax')
])

In [ ]:
#Step 4: We use the "checkpoint" callback to save the best model during training
from tensorflow.keras.callbacks import ModelCheckpoint
import os
from google.colab import drive

# Mount Google Drive to save checkpoints permanently
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Directory in Google Drive where checkpoints will be saved
checkpoint_dir = "/content/drive/MyDrive/Mis estudios /BBS/Woxsen/checkpoints_newdataset"
os.makedirs(checkpoint_dir, exist_ok=True)

# Path to save the best checkpoint

checkpoint_filepath = os.path.join(checkpoint_dir, "newData_best_alzheimers_model.h5")`

# Callback to save the best model during training
checkpoint = ModelCheckpoint(
    filepath=checkpoint_filepath,
    save_weights_only=False,   # saves the entire model (architecture + weights)
    monitor="val_accuracy",    # metric to monitor
    mode="max",                # "higher is better" for accuracy
    save_best_only=True,       # save only when validation accuracy improves
    verbose=1
)




In [17]:
# --- Section 4: Compile and Train with Checkpoint ---

# Compile the model before training
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',  # Suitable loss function for multi-class classification
    metrics=['accuracy']              # Track accuracy during training
)

# Print a summary of the model architecture
model.summary()

# Callback to save the best model during training
checkpoint = ModelCheckpoint(
    filepath=checkpoint_filepath,
    save_weights_only=False,   # saves the entire model (architecture + weights)
    monitor="val_accuracy",    # metric to monitor
    mode="max",                # "higher is better" for accuracy
    save_best_only=True,       # save only when validation accuracy improves
    verbose=1
)

# Train the model with checkpoint
history = model.fit(
    train_generator,
    epochs=epochs,
    validation_data=validation_generator,
    callbacks=[checkpoint]
)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 512)            │    44,302,848 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 4)              │         2,052 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 44,398,148 (169.37 MB)

 Trainable params: 44,398,148 (169.37 MB)

 Non-trainable params: 0 (0.00 B)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/15
256/256 ━━━━━━━━━━━━━━━━━━━━ 0s 401ms/step - accuracy: 0.2795 - loss: 1.6433
Epoch 1: val_accuracy improved from -inf to 0.37256, saving model to /content/drive/MyDrive/Mis estudios /BBS/Woxsen/checkpoints_newdataset/newData_best_alzheimers_model.h5


256/256 ━━━━━━━━━━━━━━━━━━━━ 158s 585ms/step - accuracy: 0.2795 - loss: 1.6424 - val_accuracy: 0.3726 - val_loss: 1.3055
Epoch 2/15
256/256 ━━━━━━━━━━━━━━━━━━━━ 0s 406ms/step - accuracy: 0.4136 - loss: 1.2490
Epoch 2: val_accuracy improved from 0.37256 to 0.69336, saving model to /content/drive/MyDrive/Mis estudios /BBS/Woxsen/checkpoints_newdataset/newData_best_alzheimers_model.h5


256/256 ━━━━━━━━━━━━━━━━━━━━ 134s 525ms/step - accuracy: 0.4138 - loss: 1.2487 - val_accuracy: 0.6934 - val_loss: 0.7776
Epoch 3/15
256/256 ━━━━━━━━━━━━━━━━━━━━ 0s 448ms/step - accuracy: 0.6222 - loss: 0.8395
Epoch 3: val_accuracy improved from 0.69336 to 0.75879, saving model to /content/drive/MyDrive/Mis estudios /BBS/Woxsen/checkpoints_newdataset/newData_best_alzheimers_model.h5


256/256 ━━━━━━━━━━━━━━━━━━━━ 154s 574ms/step - accuracy: 0.6222 - loss: 0.8393 - val_accuracy: 0.7588 - val_loss: 0.5903
Epoch 4/15
256/256 ━━━━━━━━━━━━━━━━━━━━ 0s 388ms/step - accuracy: 0.6880 - loss: 0.6902
Epoch 4: val_accuracy did not improve from 0.75879
256/256 ━━━━━━━━━━━━━━━━━━━━ 178s 482ms/step - accuracy: 0.6880 - loss: 0.6902 - val_accuracy: 0.7080 - val_loss: 0.5741
Epoch 5/15
256/256 ━━━━━━━━━━━━━━━━━━━━ 0s 396ms/step - accuracy: 0.7064 - loss: 0.6220
Epoch 5: val_accuracy did not improve from 0.75879
256/256 ━━━━━━━━━━━━━━━━━━━━ 125s 488ms/step - accuracy: 0.7064 - loss: 0.6220 - val_accuracy: 0.7061 - val_loss: 0.5733
Epoch 6/15
256/256 ━━━━━━━━━━━━━━━━━━━━ 0s 392ms/step - accuracy: 0.7361 - loss: 0.5897
Epoch 6: val_accuracy did not improve from 0.75879
256/256 ━━━━━━━━━━━━━━━━━━━━ 139s 478ms/step - accuracy: 0.7361 - loss: 0.5897 - val_accuracy: 0.6924 - val_loss: 0.5694
Epoch 7/15
256/256 ━━━━━━━━━━━━━━━━━━━━ 0s 399ms/step - accuracy: 0.7363 - loss: 0.5811
Epoch 7: va

## Second architecture
Aims to improve the results achieved with the first one, by loading the previous model and trying another architectural approach


In [9]:
# --- Section 3: Build the Multi-Layer CNN Model ---

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization

model = Sequential([
    # First Convolutional Block (BatchNorm added)
    Conv2D(32, (3, 3), activation='relu', input_shape=(img_height, img_width, 3)),
    BatchNormalization(),  # Purpose: stabilize the training
    MaxPooling2D(pool_size=(2, 2)),

    # Second Convolutional Block (number of filters rose from 64 -> 128)
    Conv2D(128, (3, 3), activation='relu'),  # before 64, ahora 128
    BatchNormalization(),  # NUEVO
    MaxPooling2D(pool_size=(2, 2)),

    # Third Convolutional Block (number of filters rose from 128 → 256)
    Conv2D(256, (3, 3), activation='relu'),
    BatchNormalization(),  # NUEVO
    MaxPooling2D(pool_size=(2, 2)),

    # Fourth Convolutional Block (New block with 512 filtros)
    Conv2D(512, (3, 3), activation='relu'),  # New convolutional block
    BatchNormalization(),
    MaxPooling2D(pool_size=(2, 2)),

    # Flatten the 3D output to 1D
    Flatten(),

    # Fully Connected Layers
    Dense(1024, activation='relu'),  # CHANGE: before 512, now 1024 neurons
    Dropout(0.5), # Dropout for reducing overfitting

    Dense(num_classes, activation='softmax')  # final output with softmax for multiclass classiification
])


### Compile and Train the Model (and a decision about it)

But, since the model was not outperforming the previous one nor improving its results, we decided to stop its training and save both time and resources. In a more complex experiment and in a research project with greater objectives and resources, this decision would not be taken, but for this project, since it's in a scholar level, it was the proper call.

In [ ]:

from tensorflow.keras.models import load_model
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping

# --- Load previously saved model (if exists) ---
# If a checkpoint already exists, load it to continue training
try:
    model = load_model("/content/drive/MyDrive/Mis estudios /BBS/Woxsen/checkpoints_newdataset/newData_best_alzheimers_model.h5")
    print("✅ Model loaded from 'alzheimers_checkpoint.h5'. Continuing training...")
except:
    print("⚠️ No checkpoint found. Training from scratch.")

# Compile the model (it must be recompiled even if loaded from checkpoint)
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',  # Loss function for multi-class classification
    metrics=['accuracy']
)

# Print a summary of the model architecture
model.summary()

# --- Checkpoint Callback ---
# This will save the best model based on validation accuracy
checkpoint_filepath = "alzheimers_checkpoint.h5"

checkpoint = ModelCheckpoint(
    filepath=checkpoint_filepath,
    save_weights_only=False,   # saves the entire model (architecture + weights)
    monitor="val_accuracy",    # evaluation metric
    mode="max",
    save_best_only=True,       # saves only when it improves
    verbose=1
)

# Reduce learning rate when a metric has stopped improving
reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",        # monitor validation loss
    factor=0.2,                # reduce LR by this factor
    patience=5,                # number of epochs with no improvement before reducing
    verbose=1,
    min_lr=1e-6                # lower bound on the learning rate
)

# Stop training when validation loss does not improve after patience epochs
early_stopping = EarlyStopping(
    monitor="val_loss",        # monitor validation loss
    patience=8,                # stop after 8 epochs with no improvement
    verbose=1,
    restore_best_weights=True  # restore the best weights at the end
)


# Train the model using the data generators with all callbacks
history = model.fit(
    train_generator,
    epochs=epochs,
    validation_data=validation_generator,
    callbacks=[checkpoint, reduce_lr, early_stopping]
)

print(f"\nTraining finished. The best model was saved at: {checkpoint_filepath}")

✅ Model loaded from 'alzheimers_checkpoint.h5'. Continuing training...


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 512)            │    44,302,848 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 4)              │         2,052 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 44,398,148 (169.37 MB)

 Trainable params: 44,398,148 (169.37 MB)

...

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/25
256/256 ━━━━━━━━━━━━━━━━━━━━ 0s 594ms/step - accuracy: 0.6891 - loss: 0.6918
Epoch 1: val_accuracy improved from -inf to 0.69824, saving model to alzheimers_checkpoint.h5


256/256 ━━━━━━━━━━━━━━━━━━━━ 207s 777ms/step - accuracy: 0.6892 - loss: 0.6917 - val_accuracy: 0.6982 - val_loss: 0.5856 - learning_rate: 0.0010
Epoch 2/25
256/256 ━━━━━━━━━━━━━━━━━━━━ 0s 440ms/step - accuracy: 0.7171 - loss: 0.6308
Epoch 2: val_accuracy improved from 0.69824 to 0.70801, saving model to alzheimers_checkpoint.h5


256/256 ━━━━━━━━━━━━━━━━━━━━ 149s 581ms/step - accuracy: 0.7171 - loss: 0.6308 - val_accuracy: 0.7080 - val_loss: 0.5472 - learning_rate: 0.0010
Epoch 3/25
256/256 ━━━━━━━━━━━━━━━━━━━━ 0s 434ms/step - accuracy: 0.7208 - loss: 0.6101
Epoch 3: val_accuracy did not improve from 0.70801
256/256 ━━━━━━━━━━━━━━━━━━━━ 138s 538ms/step - accuracy: 0.7209 - loss: 0.6101 - val_accuracy: 0.7017 - val_loss: 0.5609 - learning_rate: 0.0010
Epoch 4/25
256/256 ━━━━━━━━━━━━━━━━━━━━ 0s 445ms/step - accuracy: 0.7297 - loss: 0.5900
Epoch 4: val_accuracy did not improve from 0.70801
256/256 ━━━━━━━━━━━━━━━━━━━━ 140s 547ms/step - accuracy: 0.7297 - loss: 0.5900 - val_accuracy: 0.7031 - val_loss: 0.5620 - learning_rate: 0.0010
Epoch 5/25
256/256 ━━━━━━━━━━━━━━━━━━━━ 0s 432ms/step - accuracy: 0.7429 - loss: 0.5664
Epoch 5: val_accuracy did not improve from 0.70801
256/256 ━━━━━━━━━━━━━━━━━━━━ 137s 534ms/step - accuracy: 0.7428 - loss: 0.5664 - val_accuracy: 0.6704 - val_loss: 0.6164 - learning_rate: 0.0010
Epo

KeyboardInterrupt: 

##  Make a Prediction (Example)
Here we test the model using an image from the test set

In [ ]:
# Get a batch of images and labels from the validation generator
test_images, test_labels = next(validation_generator)
test_image = test_images[0]
true_label_index = np.argmax(test_labels[0])

# Expand dimensions to create a batch of 1 image
test_image = np.expand_dims(test_image, axis=0)

# Make a prediction
predictions = model.predict(test_image)
predicted_class_index = np.argmax(predictions)

print("\n--- Prediction Example ---")
print(f"True Class: {class_names[true_label_index]}")
print(f"Predicted Class: {class_names[predicted_class_index]}")
print(f"Prediction Probabilities: {predictions[0]}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step

--- Prediction Example ---
True Class: Moderate Impairment
Predicted Class: Moderate Impairment
Prediction Probabilities: [2.23305665e-06 9.99978185e-01 1.17043755e-05 7.89542992e-06]
